# 🤖 Autonomous AI Software Engineer

A portfolio-ready agentic AI prototype that turns a software task into a plan, analyzes a repository, proposes a patch, validates it with tests, performs a critic review, and requires human approval before external actions.

> Core notebook runs locally without a paid LLM API. Production extensions can connect GitHub, MCP tools, sandboxed execution, and an LLM.


## 🎯 Architecture

```text
Developer Issue
      ↓
🧭 Planner Agent
      ↓
🗂️ Repository Analyzer
      ↓
✍️ Code Agent
      ↓
🧪 Test Agent
      ↓
🔎 Critic Agent
      ↓
🛡️ Human Approval
      ↓
🐙 Git / GitHub / MCP
```


In [ ]:
from dataclasses import dataclass, field
from pathlib import Path
import re, json

@dataclass
class AgentState:
    task: str
    files: list[str] = field(default_factory=list)
    plan: list[str] = field(default_factory=list)
    proposed_changes: list[str] = field(default_factory=list)
    tests: list[str] = field(default_factory=list)
    review_findings: list[str] = field(default_factory=list)
    approved: bool = False

def route_task(task):
    t = task.lower()
    if any(x in t for x in ["bug", "error", "fix", "failure"]): return "bug_fix"
    if any(x in t for x in ["test", "coverage"]): return "testing"
    if any(x in t for x in ["refactor", "cleanup", "architecture"]): return "refactor"
    return "feature"

task = "Add input validation to user registration and create tests for invalid email addresses."
state = AgentState(task)
print("Task type:", route_task(task))


## 🗂️ 1. Repository Analyzer

In [ ]:
sample_repo = {
    "src/auth/register.py": "def register_user(email, password):\n    return {'email': email, 'password': password}\n",
    "tests/test_register.py": "def test_register_user():\n    assert True\n",
    "README.md": "# Sample Application\n",
}

def analyze_repository(repo):
    files = list(repo)
    languages = {}
    for f in files:
        ext = Path(f).suffix or "none"
        languages[ext] = languages.get(ext, 0) + 1
    return {"files": files, "languages": languages}

repo_info = analyze_repository(sample_repo)
state.files = repo_info["files"]
print(repo_info)


## 🧠 2. Planning Agent

In [ ]:
def create_plan(task):
    kind = route_task(task)
    if kind == "bug_fix":
        return ["Reproduce failure", "Trace root cause", "Implement minimal fix",
                "Add regression test", "Run test suite"]
    if kind == "testing":
        return ["Inspect coverage", "Identify missing branches", "Add focused tests",
                "Run suite", "Summarize results"]
    if kind == "refactor":
        return ["Inspect architecture", "Identify coupling", "Design low-risk refactor",
                "Refactor incrementally", "Run tests"]
    return ["Inspect implementation", "Define behavior and validation rules",
            "Implement smallest safe change", "Add positive and negative tests",
            "Run validation and review"]

state.plan = create_plan(state.task)
for i, step in enumerate(state.plan, 1):
    print(f"{i}. {step}")


## ✍️ 3. Code Agent — Structured Patch Proposal

In [ ]:
patch = '''--- a/src/auth/register.py
+++ b/src/auth/register.py
@@
+import re
+EMAIL_PATTERN = re.compile(r"^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$")
+
 def register_user(email, password):
+    if not EMAIL_PATTERN.match(email):
+        raise ValueError("Invalid email address")
     return {"email": email, "password": password}
'''

state.proposed_changes = [
    "Add reusable email validation",
    "Reject invalid email addresses",
    "Add regression tests",
]
print(patch)


## 🧪 4. Test Agent

In [ ]:
def validate_email(email):
    return bool(re.match(r"^[^@\\s]+@[^@\\s]+\\.[^@\\s]+$", email))

test_cases = {
    "developer@example.com": True,
    "person.name@company.co.in": True,
    "invalid-email": False,
    "missing-domain@": False,
    "@missing-user.com": False,
}

results = []
for email, expected in test_cases.items():
    actual = validate_email(email)
    results.append((email, actual, expected, actual == expected))

assert all(ok for _, _, _, ok in results)
state.tests = [f"{e}: {'PASS' if ok else 'FAIL'}" for e, _, _, ok in results]
for row in results: print(row)
print("All validation tests passed ✅")


## 🔎 5. Critic Agent

In [ ]:
def review_patch(patch):
    findings = []
    if "password" in patch.lower():
        findings.append("⚠️ Review credential handling; never log or persist plaintext passwords.")
    if "EMAIL_PATTERN" in patch:
        findings.append("✅ Validation logic is centralized and readable.")
    findings.append("🧪 Ensure the behavior change has regression tests.")
    return findings

state.review_findings = review_patch(patch)
for finding in state.review_findings:
    print(finding)


## 🛡️ 6. Human Approval Gate

In [ ]:
def approval_gate(state, approve=False):
    state.approved = approve
    return ("✅ Approved for controlled integration."
            if approve else
            "⏸️ Staged for human review; no external action is authorized.")

print(approval_gate(state, approve=False))


## 📦 7. Final Engineering Report

In [ ]:
report = {
    "task": state.task,
    "task_type": route_task(state.task),
    "files_analyzed": state.files,
    "plan": state.plan,
    "proposed_changes": state.proposed_changes,
    "tests": state.tests,
    "review_findings": state.review_findings,
    "approved_for_external_action": state.approved,
}
print(json.dumps(report, indent=2))


## 🤖 Optional LLM Extension

Replace the deterministic planner/coder/critic with an OpenAI-compatible LLM. Keep API keys in environment variables.

Recommended agent roles:

- 🧭 Planner — decomposes issues into verifiable subtasks
- 🗂️ Repository Agent — retrieves relevant code context
- ✍️ Coding Agent — generates minimal patches
- 🧪 Test Agent — writes/runs tests
- 🔎 Critic — checks correctness, security, and maintainability
- 📝 Final Agent — creates a PR-ready summary

For large repositories, add **Code RAG** so the model retrieves only relevant files/functions.


## 🔌 Production Tool Layer

Potential integrations:

- 🐙 GitHub — issues, branches, commits, pull requests
- 📁 Filesystem — isolated workspace access
- 🧪 Test runners — pytest, npm test, etc.
- 🔍 Static analysis — Ruff, ESLint, mypy, Semgrep
- 🐳 Sandbox — isolated execution
- 🔌 MCP — standardized engineering tools

Generated code should execute in a sandbox with timeouts, restricted filesystem/network access, and human approval for commits, deployment, deletion, or production access.


## 📏 Evaluation Framework

| Metric | Measure |
|---|---|
| 🎯 Task success | Correctly solves requested issue |
| 🧪 Test pass rate | Generated changes passing tests |
| 🔁 Repair rate | Failed changes successfully repaired |
| 🧠 Plan quality | Quality of task decomposition |
| 🛡️ Safety | Unauthorized/unsafe action rate |
| ⏱️ Efficiency | Latency and tool calls |
| 💰 Cost | Tokens/API/tool execution |
| 👨‍💻 PR acceptance | Developer acceptance rate |


## 🗺️ Roadmap

- [x] Task routing
- [x] Repository analysis
- [x] Planning agent
- [x] Patch proposal
- [x] Test agent
- [x] Critic agent
- [x] Human approval gate
- [ ] Real Git integration
- [ ] GitHub Issues + PRs
- [ ] LangGraph state machine
- [ ] MCP engineering tools
- [ ] Sandboxed code execution
- [ ] Automatic test-failure repair loop
- [ ] Code RAG
- [ ] Agent tracing and evaluation dashboard
- [ ] React/Streamlit web UI


## 💼 Resume Description

> Built an autonomous AI software-engineering agent that analyzes developer tasks and repositories, generates implementation plans and code patches, validates changes through automated tests, performs AI-assisted code review, and uses human approval gates for safe Git/GitHub workflows.
